In [1]:
# diagnostic_matches.py
import os
import csv
import unicodedata
import re
from rapidfuzz import fuzz
import firebase_admin
from firebase_admin import credentials, firestore

# === CONFIGURE AQUI ===
CREDENTIALS_PATH = '../../private_key.json'
IMAGES_FOLDER = r'C:\Users\Layanny\Documents\patrimonygo\src\assets\Patrimonios'
FIRESTORE_COLLECTION = 'patrimonios_santos'
TOP_N = 5  # quantas sugestões mostrar por grupo
# ======================

# init firebase
cred = credentials.Certificate(CREDENTIALS_PATH)
firebase_admin.initialize_app(cred)
db = firestore.client()

def normalize(text: str) -> str:
    if text is None:
        return ''
    text = re.sub(r'\.[A-Za-z0-9]+$', '', text)      # remove extensão se houver
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r'[^0-9A-Za-z\s]', '', text)       # remove pontuação (mantém espaços)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

def group_images(folder):
    groups = {}
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.gif')):
            continue
        # tentativa de extrair base: remove última parte que costuma ser número
        # Ex: "ALFÂNDEGA DA ... 01.jpg" -> "ALFÂNDEGA DA ..."
        base_raw = fname.rsplit(' ', 1)[0]
        if base_raw == fname:
            base_raw = os.path.splitext(fname)[0]
        groups.setdefault(base_raw, []).append(fname)
    return groups

def build_index():
    docs = list(db.collection(FIRESTORE_COLLECTION).stream())
    index = {}
    for d in docs:
        data = d.to_dict()
        name = data.get('name') or data.get('title') or ''
        norm = normalize(name)
        # se houver duplicatas normalizadas, guardamos lista (defensivo)
        index.setdefault(norm, []).append((d.id, name))
    return index

def list_firestore_names():
    docs = list(db.collection(FIRESTORE_COLLECTION).stream())
    names = []
    for d in docs:
        data = d.to_dict()
        name = data.get('name') or data.get('title') or ''
        names.append((d.id, name, normalize(name)))
    return names

def diagnostic():
    groups = group_images(IMAGES_FOLDER)
    print(f'Grupos detectados: {len(groups)}')
    fs_names = list_firestore_names()
    fs_norms = [n for (_,_,n) in fs_names]

    out_rows = []
    for base_raw, files in groups.items():
        base_norm = normalize(base_raw)
        # compute scores against every firestore normalized name
        scores = []
        for doc_id, orig_name, norm_name in fs_names:
            score = fuzz.ratio(base_norm, norm_name)
            scores.append((score, doc_id, orig_name, norm_name))
        scores.sort(reverse=True, key=lambda x: x[0])
        best = scores[:TOP_N]
        print('\n----------------------------------------')
        print(f'Grupo de arquivo: "{base_raw}"  (normalizado: "{base_norm}")\nArquivos: {files}')
        print('Melhores correspondências (score, doc_id, firestore_name):')
        for score, doc_id, orig_name, norm_name in best:
            print(f'  {int(score):3d}  {doc_id}  "{orig_name}"  (norm: "{norm_name}")')
            out_rows.append([base_raw, ','.join(files), base_norm, score, doc_id, orig_name, norm_name])
    # salvar CSV com sugestões
    with open('suggestions.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['group_base', 'files', 'group_norm', 'score', 'doc_id', 'firestore_name', 'firestore_norm'])
        writer.writerows(out_rows)
    print('\nDone. Arquivo suggestions.csv gerado — abra e revise as sugestões.')
    print('Se quiser, podemos: baixar matches com score>=X automaticamente, reduzir threshold, ou criar um mapeamento manual (CSV) para aplicar atualizações.')
    
if __name__ == '__main__':
    diagnostic()


Grupos detectados: 25

----------------------------------------
Grupo de arquivo: "ALFÂNDEGA DA RECEITA FEDERAL DO BRASIL EM SANTOS"  (normalizado: "alfandega da receita federal do brasil em santos")
Arquivos: ['ALFÂNDEGA DA RECEITA FEDERAL DO BRASIL EM SANTOS 01.jpg', 'ALFÂNDEGA DA RECEITA FEDERAL DO BRASIL EM SANTOS 01.png', 'ALFÂNDEGA DA RECEITA FEDERAL DO BRASIL EM SANTOS 03.jpg']
Melhores correspondências (score, doc_id, firestore_name):
    0  6A4BYFRKJ6F2fVTMCJph  ""  (norm: "")
    0  7RxJEue6YG0SBl44V1hu  ""  (norm: "")
    0  CdDBZovnsM0ur1bjn6hj  ""  (norm: "")
    0  Dd4WTvBZiDCZykQFsMuj  ""  (norm: "")
    0  F7vmtLH41RBzI28pRCQ7  ""  (norm: "")

----------------------------------------
Grupo de arquivo: "BASÍLICA EMBARÉ"  (normalizado: "basilica embare")
Arquivos: ['BASÍLICA EMBARÉ 01.jpg', 'BASÍLICA EMBARÉ 02.jpg', 'BASÍLICA EMBARÉ 03.jpg']
Melhores correspondências (score, doc_id, firestore_name):
    0  6A4BYFRKJ6F2fVTMCJph  ""  (norm: "")
    0  7RxJEue6YG0SBl44V1hu  